# 01 — Event-level cleaning and camera-aware split (v2)
**Wildlife confidence calibration | New experiment, separate from the completed v1 study**

**Why this replaces the old Notebook 01:** Previously, cameras were allocated before event-level cleaning and the five-category selection used the resulting development counts. Because camera event volumes and species differ greatly, the validation, calibration and policy sets became dominated by individual cameras. Here we **clean images and sequences → choose one reproducible frame per event → review categories → freeze five categories → optimize and verify camera-disjoint splits**.

The initial five categories (`coyote`, `deer`, `fox`, `rabbit`, `skunk`) are deliberately kept **for continuity only**, **not** because they were the five most frequent. Change `SELECTED_CLASSES` below only **before the first frozen run**, if you decide on another documented category rule. This uses category metadata, never model scores or test performance.

**Run in a new working directory.** This writes `wildlife_confidence_project_v2/`, not the completed `wildlife_confidence_project/`. Original v1 outputs/results must remain unchanged. Run Notebook 02 v2 after this. Downstream notebooks 03–05 must be separately configured and rerun for v2; do not reuse the old model checkpoints or calibration results.

## 1. Configuration and dependencies
Install the existing lightweight packages if necessary: `python -m pip install numpy pandas matplotlib pillow`. The experiment uses only NumPy/Pandas for the split optimizer; it does **not** require scikit-learn or PyTorch here.

Preset thresholds are **design choices**, not guarantees of future performance. To choose different thresholds, review feasibility and declare a new experiment before final testing. The optimizer never examines model outcomes.

In [1]:
from pathlib import Path
from urllib.parse import quote
from urllib.request import urlopen
import hashlib, json, math, platform, sys, time, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = None  # e.g. Path('/home/me/project/wildlife_confidence_project_v2')
ROOT = (Path.cwd() / 'wildlife_confidence_project_v2') if PROJECT_ROOT is None else Path(PROJECT_ROOT).expanduser()
ROOT = ROOT.resolve()
RAW, PREPARED, PREVIEW = (ROOT / 'data/raw', ROOT / 'data/prepared', ROOT / 'data/preview_images')
for directory in (RAW, PREPARED, PREVIEW):
    directory.mkdir(parents=True, exist_ok=True)

EXPERIMENT_ID = 'five_categories_camera_balanced_v2'
SEED = 42
SELECTED_CLASSES = ['coyote', 'deer', 'fox', 'rabbit', 'skunk']  # Explicit, pre-split scope choice.
METADATA_URL = 'https://storage.googleapis.com/public-datasets-lila/caltechcameratraps/labels/caltech_camera_traps.json.zip'
IMAGE_BASE_URL = 'https://lilawildlife.blob.core.windows.net/lila-wildlife/caltech-unzipped/cct_images/'
TRAIN_PER_CLASS_CAP = 1200
EVAL_BUDGETS = {'validation': 1500, 'calibration': 1500, 'policy': 1500, 'test': 3000}
SPLITS = ['train', 'validation', 'calibration', 'policy', 'test']
SPLIT_FRACTIONS = {'train': .50, 'validation': .10, 'calibration': .10, 'policy': .10, 'test': .20}
MAX_EVAL_CAMERA_SHARE = .20  # Notebook 02 cap (validation/calibration/policy only).
MAX_POOL_CAMERA_SHARE = {'train': .30, 'validation': .27, 'calibration': .27,
                         'policy': .27, 'test': .30}
MIN_CLASS_EVENTS_PER_EVAL_SPLIT = 75
MIN_CAMERAS_PER_CLASS = {'train': 6, 'validation': 3, 'calibration': 3,
                         'policy': 3, 'test': 3}
SEARCH_RESTARTS, SEARCH_STEPS = 10, 14000
DOWNLOAD_PREVIEW = False
if len(SELECTED_CLASSES) != 5 or len(set(SELECTED_CLASSES)) != 5:
    raise ValueError('Choose exactly five distinct category names before splitting.')

def sha_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def stable_hash(value, purpose):
    return hashlib.sha256(f'{SEED}|{purpose}|{value}'.encode('utf-8')).hexdigest()

def freeze_file(path, content):
    path=Path(path)
    if path.exists():
        if path.read_text(encoding='utf-8') != content:
            raise ValueError(f'Frozen v2 artifact differs: {path}. Use a NEW experiment, do not overwrite.')
        return
    temp=Path(str(path)+'.part')
    temp.write_text(content, encoding='utf-8')
    temp.replace(path)

def json_text(value):
    return json.dumps(value, indent=2, sort_keys=True, ensure_ascii=False)+'\n'

print('New v2 project:', ROOT)
print('Python:',sys.version.split()[0], 'NumPy:', np.__version__, 'Pandas:', pd.__version__)

New v2 project: /home/fia/Schreibtisch/wildlife_confidence_project_v2/wildlife_confidence_project_v2
Python: 3.11.16 NumPy: 2.4.6 Pandas: 3.0.5


## 2. Verify source metadata and individual image labels
Read the official metadata archive (downloaded if absent). Repeated annotation records for the same category are allowed. Images with **multiple distinct categories** or missing labels are excluded and logged. We never arbitrarily assign one of several conflicting labels.

In [2]:
def download_atomic(url, dest):
    dest=Path(dest)
    if dest.exists():
        return dest
    tmp=Path(str(dest)+'.part')
    try:
        with urlopen(url,timeout=60) as resp, tmp.open('wb') as f:
            for part in iter(lambda:resp.read(1024*1024),b''):
                f.write(part)
        tmp.replace(dest)
    finally:
        tmp.unlink(missing_ok=True)
    return dest

archive=download_atomic(METADATA_URL, RAW/'caltech_camera_traps.json.zip')
with zipfile.ZipFile(archive) as z:
    members=[s for s in z.namelist() if s.endswith('.json')]
    if len(members)!=1:
        raise ValueError('Expected exactly one JSON in the CCT metadata ZIP.')
    metadata=json.loads(z.read(members[0]))
metadata_sha256=sha_file(archive)
for key in ['images','annotations','categories']:
    if key not in metadata:
        raise ValueError(f'Missing metadata key {key}')
images=pd.DataFrame(metadata['images'])
ann=pd.DataFrame(metadata['annotations'])
categories=pd.DataFrame(metadata['categories'])
required={'id','file_name','location','seq_id'}
if not required.issubset(images) or images['id'].duplicated().any() or images.file_name.duplicated().any():
    raise ValueError('Invalid/missing/duplicate image ID, filename or camera/sequence columns.')
if images[['location','seq_id']].isna().any().any() or images[['location','seq_id']].astype(str).eq('').any().any():
    raise ValueError('Missing camera or sequence identifiers.')
if categories.id.duplicated().any() or categories.name.duplicated().any():
    raise ValueError('Category IDs/names must be unique.')
if not ann.image_id.isin(images.id).all() or not ann.category_id.isin(categories.id).all():
    raise ValueError('Broken annotation foreign keys.')
images['location']=images['location'].astype(str)
images['seq_id']=images['seq_id'].astype(str)
if images.groupby('seq_id').location.nunique().gt(1).any():
    raise ValueError('A sequence is associated with multiple cameras: investigate event identity.')
category_names=dict(zip(categories.id,categories.name))
if not set(SELECTED_CLASSES).issubset(set(categories.name)):
    raise ValueError('SELECTED_CLASSES includes names absent from the source metadata.')
label_sets=ann.groupby('image_id').category_id.agg(lambda s:tuple(sorted(set(s))))
catalogue=images.rename(columns={'id':'image_id'}).copy()
catalogue['category_ids']=catalogue.image_id.map(label_sets).map(lambda x:x if isinstance(x,tuple) else ())
catalogue['n_distinct_labels']=catalogue.category_ids.map(len)
catalogue['exclusion_reason']=np.select([catalogue.n_distinct_labels.eq(0),catalogue.n_distinct_labels.gt(1)],
                                         ['no_label','multiple_distinct_labels'],default='')
excluded=catalogue[catalogue.exclusion_reason.ne('')].copy()
clean_images=catalogue[catalogue.exclusion_reason.eq('')].copy()
clean_images['category_id']=clean_images.category_ids.map(lambda t:t[0])
clean_images['class_name']=clean_images.category_id.map(category_names)
if clean_images.class_name.isna().any():
    raise ValueError('Unable to map category IDs to names.')
freeze_file(PREPARED/'excluded_images.csv', excluded.to_csv(index=False))
print(f'{len(images):,} original photographs; {len(excluded):,} image-level exclusions; '
      f'{len(clean_images):,} eligible single-label photographs.')
print('Original metadata SHA-256:',metadata_sha256)

243,100 original photographs; 77 image-level exclusions; 243,023 eligible single-label photographs.
Original metadata SHA-256: 976bd9bc9380df7a03018ac70b5340bd2ff81d79a99271058f026a539450d1d6


## 3. Resolve sequence-level ambiguity and freeze one image per event **before splitting**
An event (`seq_id`) is excluded if any of its images had an ambiguous/missing annotation, or its individually single-labelled images disagree on category. Then one representative photograph is chosen using a deterministic SHA-256 rank **without using image clarity or classifier results**. This event-level choice is made for all categories, not based on split membership.

**Caution:** One trigger event is not necessarily an independent animal encounter. Camera-level correlation remains possible. This stage prevents repeated frames from being separately counted, but cannot establish semantic crop/label correctness.

In [3]:
seq_categories=clean_images.groupby('seq_id').category_id.nunique()
mixed=set(seq_categories[seq_categories.gt(1)].index)
contains_excluded=set(excluded.seq_id)
ambiguous=mixed|contains_excluded
excluded_seq=pd.DataFrame([{'seq_id':sid,
    'mixed_category_in_single_label_images':sid in mixed,
    'contains_excluded_image':sid in contains_excluded} for sid in sorted(ambiguous)],
    columns=['seq_id','mixed_category_in_single_label_images','contains_excluded_image'])
freeze_file(PREPARED/'excluded_sequences.csv',excluded_seq.to_csv(index=False))
unambiguous=clean_images.loc[~clean_images.seq_id.isin(ambiguous)].copy()
unambiguous['frame_rank']=unambiguous.image_id.map(lambda s:stable_hash(s,'frame'))
events=(unambiguous.sort_values(['frame_rank','image_id'])
       .drop_duplicates('seq_id').copy())
events['sampling_rank']=events.image_id.map(lambda s:stable_hash(s,'subset'))
assert events.image_id.is_unique and events.seq_id.is_unique
EVENT_COLUMNS=['image_id','file_name','location','seq_id','category_id','class_name']
all_events=events[EVENT_COLUMNS].sort_values('image_id').reset_index(drop=True)
freeze_file(PREPARED/'event_manifest_all_categories.csv',all_events.to_csv(index=False))
print(f'{len(ambiguous):,} ambiguous sequence IDs excluded; {len(all_events):,} retained one-frame events.')

47 ambiguous sequence IDs excluded; 180,961 retained one-frame events.


## 4. Review **all** category support before fixing the five-category scope
The table shows *clean event counts* and *distinct cameras* (rather than photograph counts from a random earlier split). Only then should a researcher commit to the five categories.

The current preset deliberately retains the previous five categories **as a scientific scope choice**. It is **not** automatic top-five selection. To test an alternative set, edit `SELECTED_CLASSES` above **and create a fresh v2 experiment directory/ID** before running the split. Do not switch categories based on final model results.

In [4]:
photo_counts=clean_images.groupby('class_name').image_id.size().rename('single_label_photographs')
category_review=(all_events.groupby('class_name').agg(
    usable_events=('seq_id','size'), distinct_cameras=('location','nunique'))
    .join(photo_counts).fillna(0))
largest=(all_events.groupby(['class_name','location']).size()
         .groupby(level=0).max().rename('largest_camera_events'))
category_review=category_review.join(largest)
category_review['largest_camera_event_share']=(
    category_review.largest_camera_events/category_review.usable_events)
category_review['selected_for_v2']=category_review.index.isin(SELECTED_CLASSES)
category_review=category_review.sort_values(['usable_events','distinct_cameras'],ascending=False)
freeze_file(PREPARED/'category_review.csv',category_review.to_csv())
print(category_review[['usable_events','distinct_cameras','largest_camera_event_share','selected_for_v2']].to_string())
chosen=events.loc[events.class_name.isin(SELECTED_CLASSES)].copy()
if chosen.class_name.nunique()!=5:
    raise ValueError('Selected categories are not represented after cleaning.')
print(f'\nChosen: {SELECTED_CLASSES}, eligible events: {len(chosen):,}, cameras: {chosen.location.nunique()}')

               usable_events  distinct_cameras  largest_camera_event_share  selected_for_v2
class_name                                                                                 
empty                 121148               126                    0.302836            False
coyote                 11117                95                    0.167131             True
deer                    7031                62                    0.318731             True
rabbit                  6283                68                    0.112208             True
opossum                 5582                35                    0.268721            False
car                     4711                11                    0.316069            False
raccoon                 4682                49                    0.217215            False
bird                    3868               118                    0.102637            False
bobcat                  3732                81                    0.098071      

## 5. Find a **camera-disjoint, stratified** allocation using event and category counts
The optimizer allocates entire cameras, **not individual photographs**. With the source metadata and a predeclared seed, it tries multiple reproducible candidate allocations and camera swaps. Its objective combines event-volume balance, category-mixture balance, per-category camera representation, and dominant-camera penalties. It includes *hard preflight checks*: class support, evaluation sample capacity under Notebook 02's planned camera cap, and maximum camera contribution to each split's eligible event pool.

This uses **metadata category labels** from potential test cameras solely to design a stratified split **before any classifier is developed**. No test model scores/metrics are accessed. The subsequent test set is still unseen to downstream fitting, but the population is not a randomly sampled wildlife stream. There is no claim of statistical optimality, perfect balance or guaranteed 5% operational error. It remains the researcher's responsibility to inspect the resulting diagnostic table **before freezing the experiment**.

In [5]:
camera_class=(pd.crosstab(chosen.location,chosen.class_name)
              .reindex(columns=SELECTED_CLASSES,fill_value=0).sort_index())
if camera_class.to_numpy().sum() != len(chosen):
    raise AssertionError('Camera/category matrix has lost events')
M=camera_class.to_numpy(dtype=np.int64)
camera_ids=np.asarray(camera_class.index,dtype=str)
n_cameras=len(camera_ids)
if n_cameras<50:
    raise ValueError('Too few cameras for the declared five-way design.')
cam_total=M.sum(axis=1)
presence=(M>0).astype(np.int64)
shares=np.asarray([SPLIT_FRACTIONS[s] for s in SPLITS])
target_sizes=np.rint(shares[:4]*n_cameras).astype(int)
target_sizes=np.append(target_sizes,n_cameras-target_sizes.sum())
if (target_sizes<1).any():
    raise ValueError('Allocation target has an empty split.')
ref=M.sum(axis=0)
ref_freq=ref/ref.sum()
target_events=ref.sum()*target_sizes/n_cameras
budgets=np.asarray([0]+[EVAL_BUDGETS[s] for s in SPLITS[1:]],dtype=float)
max_shares=np.asarray([MAX_POOL_CAMERA_SHARE[s] for s in SPLITS])
min_cams=np.asarray([MIN_CAMERAS_PER_CLASS[s] for s in SPLITS])
cap_per_eval_camera=np.floor(MAX_EVAL_CAMERA_SHARE*budgets).astype(int)

def allocation_stats(assignment):
    counts=np.zeros((5,5),dtype=np.float64)
    counts_presence=np.zeros((5,5),dtype=np.int64)
    np.add.at(counts,assignment,M)
    np.add.at(counts_presence,assignment,presence)
    volumes=counts.sum(axis=1)
    max_cam=np.asarray([cam_total[assignment==i].max() for i in range(5)])
    max_camera_frac=max_cam/volumes
    capacity=np.asarray([np.minimum(cam_total[assignment==i],cap_per_eval_camera[i]).sum()
                         for i in [1,2,3]],dtype=float)
    return counts,counts_presence,volumes,max_camera_frac,capacity

def acceptable(stat):
    ct,pres,v,share,cap=stat
    return bool((pres>=min_cams[:,None]).all()
        and (ct[1:]>=MIN_CLASS_EVENTS_PER_EVAL_SPLIT).all()
        and (share<=max_shares+1e-12).all()
        and (v[1:]>=budgets[1:]).all()
        and (cap>=budgets[1:4]).all())

def allocation_cost(assignment):
    ct,pres,v,share,cap=allocation_stats(assignment)
    vol=np.sum(((v/target_events-1)**2)*[1.4,1,1,1,1])
    mix=np.sum(((ct/v[:,None]-ref_freq)/(ref_freq+.05))**2 *np.asarray([.45,1.2,1.2,1.2,1.2])[:,None])
    gaps=np.sum(np.maximum(0,min_cams[:,None]-pres)**2)*.8
    small=np.sum((np.maximum(0,MIN_CLASS_EVENTS_PER_EVAL_SPLIT-ct[1:])/
                  MIN_CLASS_EVENTS_PER_EVAL_SPLIT)**2)*1.2
    dominant=np.sum((np.maximum(0,share-max_shares)**2)*50)
    capacity=np.sum((np.maximum(0,budgets[1:4]-cap)/1500)**2)*40
    budget=np.sum((np.maximum(0,budgets[1:]-v[1:])/1500)**2)*12
    return float(vol*2+mix*3+gaps+small+dominant+capacity+budget)

rng=np.random.default_rng(SEED)
best_score=float('inf')
best_assign=None
best_feasible_score=float('inf')
best_feasible=None
started=time.perf_counter()
for restart in range(SEARCH_RESTARTS):
    assignment=np.repeat(np.arange(5,dtype=int),target_sizes)
    rng.shuffle(assignment)
    score=allocation_cost(assignment)
    for step in range(SEARCH_STEPS):
        i,j=rng.integers(n_cameras,size=2)
        if assignment[i]==assignment[j]:
            continue
        assignment[i],assignment[j]=assignment[j],assignment[i]
        next_score=allocation_cost(assignment)
        temp=.50*max(.001,1-step/SEARCH_STEPS)**1.5
        if next_score<=score or rng.random()<math.exp(min(0,(score-next_score)/temp)):
            score=next_score
        else:
            assignment[i],assignment[j]=assignment[j],assignment[i]
        if score<best_score:
            best_score=score
            best_assign=assignment.copy()
        if score<best_feasible_score and acceptable(allocation_stats(assignment)):
            best_feasible_score=score
            best_feasible=assignment.copy()
    print(f'Restart {restart+1}/{SEARCH_RESTARTS}; best feasible score '
          f'{best_feasible_score if best_feasible is not None else "none"}',flush=True)
if best_feasible is None:
    raise RuntimeError('No camera allocation met the predeclared constraints. Review candidate '
        'class diversity, budgets, caps and criteria using metadata only; do not search based '
        'on model/test performance. Do NOT silently fall back to a random split.')
assignment=best_feasible
assignments=dict(zip(camera_ids.tolist(),[SPLITS[int(s)] for s in assignment]))
counts,support,volumes,dom_share,cap=allocation_stats(assignment)
if not acceptable((counts,support,volumes,dom_share,cap)):
    raise AssertionError('Selected assignment violated feasibility constraints.')
print('Search seconds:',round(time.perf_counter()-started,1),'| selected cost:',round(best_feasible_score,4))
print(pd.DataFrame(counts.astype(int),index=SPLITS,columns=SELECTED_CLASSES).to_string())
print('Max camera share per pool:',dict(zip(SPLITS,np.round(dom_share,3))))
print('Available events after per-camera caps (validation/calibration/policy):',cap.astype(int))

Restart 1/10; best feasible score 2.2170235520294006
Restart 2/10; best feasible score 0.16761053007552454
Restart 3/10; best feasible score 0.16761053007552454
Restart 4/10; best feasible score 0.16761053007552454
Restart 5/10; best feasible score 0.16067229291455248
Restart 6/10; best feasible score 0.16067229291455248
Restart 7/10; best feasible score 0.16067229291455248
Restart 8/10; best feasible score 0.16067229291455248
Restart 9/10; best feasible score 0.16067229291455248
Restart 10/10; best feasible score 0.16067229291455248
Search seconds: 19.5 | selected cost: 0.1607
             coyote  deer  fox  rabbit  skunk
train          5758  3630  467    3207    503
validation      866   587   84     521     91
calibration    1013   642   89     575     89
policy          915   656   82     580     99
test           2565  1516  217    1400    226
Max camera share per pool: {'train': np.float64(0.221), 'validation': np.float64(0.24), 'calibration': np.float64(0.244), 'policy': np.floa

## 6. Validate, report, and freeze the camera assignment
The allocation is accepted only if the predeclared minimums and caps hold. It is **not a mathematically guaranteed global optimum**; check the printed counts and category/camera coverage before proceeding. Further improvement based on test performance would invalidate the intended held-out evaluation.

Notebook 02 must use these same camera assignments, event IDs, five-category scope and sampling budgets. Downstream models, temperatures and thresholds from v1 **cannot** be reused.

In [6]:
chosen['split']=chosen.location.map(assignments)
assert chosen.split.notna().all()
assert chosen.groupby('location').split.nunique().eq(1).all()
assert chosen.groupby('seq_id').split.nunique().eq(1).all()
manifest_columns=['image_id','file_name','location','seq_id','category_id','class_name','split']
manifest=(chosen[manifest_columns].sort_values(['split','image_id']).reset_index(drop=True))
assert manifest.image_id.is_unique and manifest.seq_id.is_unique
camera_totals=(pd.crosstab(manifest.location,manifest.class_name)
               .reindex(columns=SELECTED_CLASSES,fill_value=0))
camera_totals['split']=camera_totals.index.map(assignments)
camera_totals['eligible_events']=camera_totals[SELECTED_CLASSES].sum(axis=1)
split_summary=manifest.groupby('split').agg(images=('image_id','size'),
    cameras=('location','nunique'),sequences=('seq_id','nunique')).reindex(SPLITS)
split_summary['largest_camera_events']=split_summary.index.map(dict(zip(SPLITS,[
    int(cam_total[assignment==idx].max()) for idx in range(5)])))
split_summary['largest_camera_share']=dom_share
class_counts=pd.crosstab(manifest.class_name,manifest.split).reindex(index=SELECTED_CLASSES,columns=SPLITS,fill_value=0)
class_cameras=(manifest.groupby(['class_name','split']).location.nunique().unstack(fill_value=0)
               .reindex(index=SELECTED_CLASSES,columns=SPLITS,fill_value=0))
freeze_file(PREPARED/'image_manifest.csv',manifest.to_csv(index=False))
freeze_file(PREPARED/'camera_category_events.csv',camera_totals.to_csv())
freeze_file(PREPARED/'class_summary.csv',category_review.to_csv())
freeze_file(PREPARED/'development_class_support.csv',class_counts.drop(columns='test').to_csv())
freeze_file(PREPARED/'class_events_by_split.csv',class_counts.to_csv())
freeze_file(PREPARED/'class_cameras_by_split.csv',class_cameras.to_csv())
freeze_file(PREPARED/'split_summary.csv',split_summary.to_csv())
split_record={
  'protocol':'camera_stratified_event_v2', 'experiment_id':EXPERIMENT_ID,
  'seed':SEED, 'metadata_sha256':metadata_sha256,
  'event_manifest_sha256':sha_file(PREPARED/'event_manifest_all_categories.csv'),
  'selected_classes':SELECTED_CLASSES,
  'assignments':assignments,
  'design': {'fractions':SPLIT_FRACTIONS,'evaluation_budgets':EVAL_BUDGETS,
   'max_evaluation_camera_share':MAX_EVAL_CAMERA_SHARE,
   'max_pool_camera_share':MAX_POOL_CAMERA_SHARE,
   'min_class_events_per_evaluation_split':MIN_CLASS_EVENTS_PER_EVAL_SPLIT,
   'min_cameras_per_class':MIN_CAMERAS_PER_CLASS,
   'optimizer':'seeded_camera_swap_search_multi_start',
   'restarts':SEARCH_RESTARTS,'steps_per_restart':SEARCH_STEPS,
   'objective_score':best_feasible_score},
}
freeze_file(PREPARED/'camera_split.json',json_text(split_record))
print('Final events & camera concentration:\n',split_summary.to_string())
print('\nSpecies event support:\n',class_counts.to_string())
print('\nSpecies camera coverage:\n',class_cameras.to_string())

Final events & camera concentration:
              images  cameras  sequences  largest_camera_events  largest_camera_share
split                                                                               
train         13565       66      13565                   2996              0.220863
validation     2149       13       2149                    516              0.240112
calibration    2408       13       2408                    587              0.243771
policy         2332       13       2332                    519              0.222556
test           5924       28       5924                   1776              0.299797

Species event support:
 split       train  validation  calibration  policy  test
class_name                                              
coyote       5758         866         1013     915  2565
deer         3630         587          642     656  1516
fox           467          84           89      82   217
rabbit       3207         521          575     580  1400


## 7. Optional training-only visual preview and provenance
Preview downloads are disabled by default; enable them to inspect a small **training-only** sample. Full crop quality auditing is handled in Notebook 02. The metadata and frozen split are complete without any image downloads.

In [7]:
preview_failures=[]
if DOWNLOAD_PREVIEW:
    sample=(manifest[manifest.split.eq('train')].groupby('class_name',group_keys=False)
            .sample(n=1,random_state=SEED).sort_values('class_name'))
    fig,axs=plt.subplots(1,len(sample),figsize=(3.1*len(sample),3.3))
    for ax,row in zip(np.atleast_1d(axs),sample.itertuples()):
        ax.axis('off')
        safe_name=f'{row.image_id}.jpg'
        destination=PREVIEW/safe_name
        try:
            url=IMAGE_BASE_URL+quote(row.file_name,safe='/')
            download_atomic(url,destination)
            with Image.open(destination) as im:
                ax.imshow(im.convert('RGB'))
            ax.set_title(f'{row.class_name} / camera {row.location}')
        except Exception as ex:
            preview_failures.append({'image_id':row.image_id,'error':str(ex)})
    fig.tight_layout();plt.show()
else:
    print('Training preview disabled (metadata preparation is complete).')
prep_record={
  'stage':'01_event_first_camera_split_v2','experiment_id':EXPERIMENT_ID,
  'seed':SEED,'metadata_url':METADATA_URL,'metadata_sha256':metadata_sha256,
  'raw_images':len(images),'annotation_records':len(ann),
  'image_level_excluded':len(excluded),'sequence_level_excluded':len(ambiguous),
  'all_clean_events':len(all_events),'selected_class_events':len(manifest),
  'selected_classes':SELECTED_CLASSES,'cameras':int(manifest.location.nunique()),
  'source_population':'five known categories, one image per unambiguous trigger sequence',
  'split_protocol':'camera_stratified_event_v2',
  'split_sha256':sha_file(PREPARED/'camera_split.json'),
  'manifest_sha256':sha_file(PREPARED/'image_manifest.csv'),
  'python':sys.version.split()[0], 'numpy':np.__version__, 'pandas':pd.__version__,
  'preview_requested':DOWNLOAD_PREVIEW, 'preview_failures':preview_failures,
}
freeze_file(PREPARED/'preparation_run.json',json_text(prep_record))
print('Stage 01 v2 complete; frozen artifacts:',PREPARED)

Training preview disabled (metadata preparation is complete).
Stage 01 v2 complete; frozen artifacts: /home/fia/Schreibtisch/wildlife_confidence_project_v2/wildlife_confidence_project_v2/data/prepared


## Continue to Notebook 02 v2
Notebook 02 reads `image_manifest.csv`, `excluded_sequences.csv` and `camera_split.json` from **this v2 project root**. It may cap how many events one camera contributes to validation/calibration/policy, but must **not** change camera assignments or categories. No classifier has been trained. Changing class selection or optimizer design later requires a newly designated experiment with fresh untouched test cameras.